In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Disable auto-scroll in notebook output for smooth interaction
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def textbook_spectrum_simulation_transparent(M):
    clear_output(wait=True)
    
    # High-resolution frequency axis covering the entire baseband and beyond
    omega = np.linspace(-3 * np.pi, 3.0 * np.pi, 2000)
    
    # Strict bandwidth setting: omega_N = 0.2 * pi.
    # For M <= 5: pi/M >= 0.2*pi (No overlap/aliasing).
    # For M >= 6: pi/M < 0.2*pi (Aliasing occurs).
    omega_N = 0.2 * np.pi  
    
    def single_triangle_spect(w, scale=1.0):
        spec = np.zeros_like(w)
        mask = np.abs(w) <= omega_N
        spec[mask] = scale * (1.0 - np.abs(w[mask]) / omega_N)
        return spec

    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # --- PLOT 1: Original Spectrum ---
    spec_orig = single_triangle_spect(omega, scale=1.0)
    axes[0].plot(omega, spec_orig, 'tab:red', linewidth=2)
    axes[0].fill_between(omega, 0, spec_orig, where=(spec_orig > 1e-5), color='tab:red', alpha=0.2)
    axes[0].axvline(x=omega_N, color='orange', linestyle='--', label=r'Bandwidth $\Omega_N = 0.2\pi$')
    axes[0].axvline(x=-omega_N, color='orange', linestyle='--')
    axes[0].set_title(r'1. Original Continuous Spectrum $\mathcal{X}(j\Omega)$', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_xlim(-2*np.pi, 2*np.pi)
    axes[0].set_ylim(-0.05, 1.2)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(loc='upper right')

    # --- PLOT 2: Periodic Replicas after Standard Sampling ---
    spec_sampled = np.zeros_like(omega)
    for k in range(-4, 5):
        spec_sampled += single_triangle_spect(omega - k * 2.0 * np.pi, scale=1.0)
        
    axes[1].plot(omega, spec_sampled, 'tab:red', linewidth=2)
    axes[1].fill_between(omega, 0, spec_sampled, where=(spec_sampled > 1e-5), color='tab:red', alpha=0.2)
    axes[1].axvline(x=np.pi, color='orange', linestyle='--', label=r'Original Nyquist Limit ($+\pi$)')
    axes[1].axvline(x=-np.pi, color='orange', linestyle='--')
    axes[1].set_title(r'2. Periodic Replicas after Sampling $\mathcal{X}(e^{j\omega})$', fontsize=10, fontweight='bold')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_xlim(-2*np.pi, 2*np.pi)
    axes[1].set_ylim(-0.05, 1.2)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(loc='upper right')

    # --- PLOT 3: Decimated Spectrum with Transparent Overlapping Replicas ---
    total_decimated_spec = np.zeros_like(omega)
    
    k_range = range(-2*M, 2*M + 1)
    for k in k_range:
        replica_center = k * (2.0 * np.pi / M)
        spec_replica = (1.0 / M) * single_triangle_spect(omega - replica_center)
        
        total_decimated_spec += spec_replica
        
        if abs(k) <= M:
            axes[2].plot(omega, spec_replica, color='tab:blue', linestyle='--', alpha=0.3, linewidth=1.0)

    # Resulting total spectrum envelope (sum of all overlapping contributions)
    axes[2].plot(omega, total_decimated_spec, color='tab:red', linewidth=2.5, label=r'Resulting Spectrum $\mathcal{X}_d(e^{j\omega})$ (Sum)')
    axes[2].fill_between(omega, 0, total_decimated_spec, where=(total_decimated_spec > 1e-5), color='tab:red', alpha=0.3)

    # Visual Warning Background: Aliasing occurs strictly for M >= 6
    has_aliasing = (omega_N > np.pi / M)
    if has_aliasing:
        axes[2].set_facecolor('#FFF0F5') # Soft pink warning background
        status_text = fr'3. Decimation Spectrum for $M = {M}$ — ALIASING DETECTED!'
    else:
        axes[2].set_facecolor('white')
        status_text = fr'3. Decimation Spectrum for $M = {M}$ — Aliasing-Free'

    axes[2].axvline(x=np.pi, color='orange', linestyle='--', linewidth=1.5, label=r'Original Nyquist ($+\pi$)')
    axes[2].axvline(x=-np.pi, color='orange', linestyle='--', linewidth=1.5, label=r'Original Nyquist ($-\pi$)')
    axes[2].axvline(x=np.pi/M, color='green', linestyle=':', linewidth=2.0, label=fr'New Limit ($\pi/M = \pi/{M}$)')
    axes[2].axvline(x=-np.pi/M, color='green', linestyle=':', linewidth=2.0)

    axes[2].set_title(status_text, fontsize=10, fontweight='bold', color='darkred' if has_aliasing else 'black')
    axes[2].set_xlabel(r'Digital Frequency ($\omega$)')
    axes[2].set_ylabel('Amplitude')
    axes[2].set_xlim(-2*np.pi, 2*np.pi)
    axes[2].set_ylim(-0.05, 1.2)
    axes[2].set_xticks([-2*np.pi, -np.pi, -np.pi/M, 0, np.pi/M, np.pi, 2*np.pi])
    axes[2].set_xticklabels([r'$-2\pi$', r'$-\pi$', r'$-\pi/M$', r'$0$', r'$\pi/M$', r'$\pi$', r'$2\pi$'])
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].legend(loc='upper right', frameon=True)

    plt.tight_layout()
    plt.show()
    
    # Print dynamic parameters
    print(f"--- Decimation Parameters for M = {M} ---")
    print(f"• Amplitude Scaling Factor (Height): 1 / M = 1 / {M} = {1.0/M:.3f}")
    print(f"• Spectral Replica Spacing (Shift): 2*pi / M = 2*pi / {M}")
    print(f"• New Digital Bandwidth Limit: pi / M = pi / {M} ≈ {np.pi/M:.3f} rad/sample")
    
    if not has_aliasing:
        print(f"Status: No Aliasing (For M={M}, bandwidth limit pi/{M} >= omega_N, replicas remain separated).")
    else:
        print(f"Status: Aliasing Occurs (For M={M}, bandwidth limit pi/{M} < omega_N, overlapping zones appear — highlighted in pink).")

# Interactive widget slider for decimation factor M up to 8
m_slider = widgets.IntSlider(
    value=2, min=1, max=8, step=1,
    description='Decimation Factor ($M$):',
    style={'description_width': 'initial'}
)

ui = widgets.VBox([m_slider])
display(ui)

out = widgets.interactive_output(textbook_spectrum_simulation_transparent, {'M': m_slider})
display(out)